In [ ]:
import os
import sys

# 1. DRIVE MOUNT AND PATHS SETUP
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory: {os.getcwd()}")
except ImportError:
    # Fallback for local execution
    DRIVE_ROOT = '.'
    print("Local execution")

# 2. SYSTEM INSTALLATIONS
print("\n Transformers and Qwen ")
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install qwen-vl-utils accelerate

!pip install "bitsandbytes>=0.46.1"

print("\n- Utility Pipeline and OpenCV ")
!pip install "opencv-python==4.10.0.84" "opencv-contrib-python==4.10.0.84" "opencv-python-headless==4.10.0.84"

print("\nInstallation complete. Restart kernel.")
print("Wait 5 seconds, then go on with the execution.")
print("=======================================================")

# 3. AUTOMATIC RESTART
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory: {os.getcwd()}")
except ImportError:
    # Fallback for local execution
    DRIVE_ROOT = '.'
    print("Local execution")

os.chdir(DRIVE_ROOT)
print(f"Directory: {os.getcwd()}")

In [ ]:
!unzip -q "/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/Videos_crop_decode.zip" -d "/content/Videos_crop_decode"

!unzip -q "/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/video_general_obj_det_finished.zip" -d "/content/video_general_obj_det_finished"

!unzip -q "/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/action_recognition_finished.zip" -d "/content/action_recognition_finished" || echo "First execution"

In [ ]:
import os
import re
import json
import glob
import torch
import gc
import time
import warnings
import numpy as np
from PIL import Image, ImageDraw
import tempfile
import shutil
import bitsandbytes as bnb
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# PATHS
# PATH_DATA_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT'
PATH_DATA_ROOT = '/content'
PATH_VIDEOS_CROP_DECODE = os.path.join(PATH_DATA_ROOT, "Videos_crop_decode")
PATH_OBJ_DET_FINISHED = os.path.join(PATH_DATA_ROOT, "video_general_obj_det_finished")
PATH_ASSOCIATIONS = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/person_object_associations'
PATH_OUTPUT_DATASET = os.path.join(PATH_DATA_ROOT, "action_recognition_finished")

# ACTIONS (TUM Kitchen: right arm / left arm actions flattened on the person)
CONSENTED_ACTIONS = [
    "CarryingWhileLocomoting",
    "TakingSomething",
    "Reaching",
    "LoweringAnObject",
    "ReleasingGraspOfSomething",
    "OpeningADrawer",
    "ClosingADrawer",
    "ClosingADoor",
    "OpeningADoor",
]

# Short definitions to help the VLM disambiguate the CamelCase labels
ACTION_DESCRIPTIONS = {
    "CarryingWhileLocomoting": "walking/moving while holding or carrying an object",
    "TakingSomething": "grasping and picking up an object (from a surface, shelf, drawer, ...)",
    "Reaching": "extending an arm/hand towards something, before touching or grasping it",
    "LoweringAnObject": "bringing a held object down to place it on a surface",
    "ReleasingGraspOfSomething": "letting go of an object that was being held",
    "OpeningADrawer": "pulling a drawer open",
    "ClosingADrawer": "pushing a drawer closed",
    "ClosingADoor": "closing a door (cupboard, fridge, ...)",
    "OpeningADoor": "opening a door (cupboard, fridge, ...)",
}

# Number of simultaneous actions to output for each window (e.g. right arm + left arm)
NUM_ACTIONS = 2

# WINDOWING
SOURCE_FPS = 30.0
WINDOW_SIZE = 60
STRIDE = 30
FRAMES_TO_SAMPLE = 20
INCLUDE_TAIL_WINDOW = True

EFFECTIVE_FPS = int(FRAMES_TO_SAMPLE / (WINDOW_SIZE / SOURCE_FPS))

# OPTIONAL CROP
# "none"  -> the model sees the full frame (keeps all the scene context)
# "actor" -> every frame of a window is cropped around the actor (same crop for the whole window)
CROP_MODE = "none"
CROP_MARGIN = 0.35
assert CROP_MODE in ("none", "actor"), "CROP_MODE must be 'none' or 'actor'"

# OBJECT NAMES IN THE PROMPT
OBJECT_NAME_FIELDS = ("label", "class", "class_name", "name", "category")
MAX_OBJECT_NAMES = 15

os.makedirs(PATH_OUTPUT_DATASET, exist_ok=True)

# MODEL LOADING
model_id = "Qwen/Qwen3-VL-8B-Instruct"

print(f"Loading model {model_id} in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_skip_modules=["visual", "lm_head"]
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)
processor = AutoProcessor.from_pretrained(model_id)
model.eval()

# Sanity check: the vision encoder must NOT contain 4-bit layers
n_visual_4bit = sum(
    1 for n, m in model.named_modules()
    if "visual" in n and isinstance(m, bnb.nn.Linear4bit)
)
print(f"[check] Linear4bit layers in the vision encoder: {n_visual_4bit} (expected 0)")
print(f"[cfg] window={WINDOW_SIZE} stride={STRIDE} sampled_frames={FRAMES_TO_SAMPLE} "
      f"fps={EFFECTIVE_FPS} tail_window={INCLUDE_TAIL_WINDOW} crop={CROP_MODE}")


# UTILITIES
def build_windows(total_frames, window_size, stride, include_tail=True):
    """
    Build the list of (start, end) windows covering the video.
    If include_tail is True and the last regular window does not reach the end of the video,
    one more window aligned to the end is added (it overlaps the previous one).
    """
    if total_frames <= window_size:
        return [(0, total_frames)]

    windows = [(s, s + window_size) for s in range(0, total_frames - window_size + 1, stride)]
    if include_tail and windows[-1][1] < total_frames:
        windows.append((total_frames - window_size, total_frames))
    return windows


def compute_actor_crop(person_boxes, margin=CROP_MARGIN):
    """
    Fixed crop (normalized x1, y1, x2, y2) for a whole window: union of the actor boxes
    plus a margin, clamped to the frame. Returns None if the actor is never visible.
    """
    valid = [b for b in person_boxes if b is not None]
    if not valid:
        return None

    x1 = min(b[0] for b in valid)
    y1 = min(b[1] for b in valid)
    x2 = max(b[2] for b in valid)
    y2 = max(b[3] for b in valid)

    mw = (x2 - x1) * margin
    mh = (y2 - y1) * margin
    crop = (max(0.0, x1 - mw), max(0.0, y1 - mh), min(1.0, x2 + mw), min(1.0, y2 + mh))

    if crop[2] <= crop[0] or crop[3] <= crop[1]:
        return None
    return crop


def get_object_name(obj_key, obj_data):
    """Return a human-readable object name from the tracking JSON entry."""
    for field in OBJECT_NAME_FIELDS:
        value = obj_data.get(field)
        if isinstance(value, str) and value.strip():
            return value.strip()
    name = re.sub(r"[_\-\s]*\d+$", "", obj_key).replace("_", " ").strip()
    return name if name else obj_key


# PERSON-OBJECT ASSOCIATIONS (output of the person_object_association step)
def load_associations(video_name):
    """Load the association file of a video, or None if it does not exist."""
    path = os.path.join(PATH_ASSOCIATIONS, f"{video_name}_associations.json")
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_associated_frames(association_data):
    """{person_id: {object_id: set of frame indices}} from the frame-level associations."""
    pairs = {}
    for frame_str, items in association_data.get("frame_associations", {}).items():
        frame = int(frame_str)
        for item in items:
            pairs.setdefault(item["person_id"], {}).setdefault(item["object_id"], set()).add(frame)
    return pairs


def _longest_consecutive(frames):
    longest = current = 0
    previous = None
    for frame in sorted(frames):
        current = current + 1 if previous is not None and frame == previous + 1 else 1
        longest = max(longest, current)
        previous = frame
    return longest


def _box_is_valid(box):
    """Same validity rule of the association step: 4 numbers, non-degenerate once clamped to [0, 1]."""
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return False
    try:
        x1, y1, x2, y2 = (min(1.0, max(0.0, float(v))) for v in box)
    except (TypeError, ValueError):
        return False
    return x1 < x2 and y1 < y2


def get_associated_objects(person_id, start_idx, end_idx, detected_objects, associated_frames,
                           min_support_ratio, min_consecutive_frames):
    """
    Ids of the objects associated to `person_id` in the window [start_idx, end_idx).
    An object is accepted with the same rule used by the association step for its own windows
    (support ratio OR consecutive frames), recomputed here on the AR window from the frame-level
    associations. In this way it also works for AR windows that do not coincide with the
    association windows (e.g. the tail window, or a different WINDOW_SIZE).
    """
    person_boxes = detected_objects[person_id].get("bbox", [])
    accepted = []
    for object_id, frames in associated_frames.get(person_id, {}).items():
        obj_data = detected_objects.get(object_id)
        if obj_data is None:
            continue
        in_window = [f for f in frames if start_idx <= f < end_idx]
        if not in_window:
            continue
        obj_boxes = obj_data.get("bbox", [])
        valid_frames = sum(
            1 for f in range(start_idx, end_idx)
            if f < len(person_boxes) and f < len(obj_boxes)
            and _box_is_valid(person_boxes[f]) and _box_is_valid(obj_boxes[f])
        )
        support_ratio = len(in_window) / max(valid_frames, 1)
        if support_ratio >= min_support_ratio or _longest_consecutive(in_window) >= min_consecutive_frames:
            accepted.append(object_id)
    return sorted(accepted)


# UTILITY FOR VISUAL PROMPTING
def draw_bboxes_on_frames(frame_paths, person_boxes, objects_boxes, temp_dir, crop_box=None):
    """
    Draw bbox: red for the target and blue for the objects.
    If crop_box (normalized x1, y1, x2, y2) is given, boxes are drawn on the full frame
    and the frame is then cropped (the same crop is applied to every frame of the window).
    """
    drawn_paths = []

    for i, path in enumerate(frame_paths):
        img = Image.open(path).convert("RGB")
        draw = ImageDraw.Draw(img)
        w, h = img.size

        # 1. Draw red box (human)
        if i < len(person_boxes) and person_boxes[i] is not None:
            px1, py1, px2, py2 = person_boxes[i]
            draw.rectangle([px1*w, py1*h, px2*w, py2*h], outline="red", width=3)

        # 2. Draw blue box (object)
        for obj_boxes in objects_boxes:
            if i < len(obj_boxes) and obj_boxes[i] is not None:
                ox1, oy1, ox2, oy2 = obj_boxes[i]
                draw.rectangle([ox1*w, oy1*h, ox2*w, oy2*h], outline="blue", width=2)

        # 3. Optional crop around the actor
        if crop_box is not None:
            img = img.crop((
                int(crop_box[0] * w), int(crop_box[1] * h),
                int(crop_box[2] * w), int(crop_box[3] * h)
            ))

        temp_path = os.path.join(temp_dir, f"frame_{i:03d}.jpg")
        img.save(temp_path)
        drawn_paths.append(temp_path)

    return drawn_paths


def _normalize(s):
    """Lowercase and keep letters only, so 'Opening a Drawer' == 'OpeningADrawer'."""
    return re.sub(r"[^a-z]", "", s.lower())


def parse_actions(output_text, action_list, max_actions=NUM_ACTIONS):
    """
    Parse the model output into a list of valid actions.
    - accepts separators like , ; | newline
    - tolerant to spaces/underscores/case differences
    - removes duplicates (preserving order) and keeps at most `max_actions`
    - returns ["unknown"] if nothing valid is found
    """
    norm_map = {_normalize(a): a for a in action_list}
    found = []
    for part in re.split(r"[,;|\n]+", output_text):
        key = _normalize(part)
        if key in norm_map and norm_map[key] not in found:
            found.append(norm_map[key])
    return found[:max_actions] if found else ["unknown"]


def process_window(drawn_frame_paths, action_list, object_names=None, max_pixels=640*480, num_actions=NUM_ACTIONS):
    """Infer the (up to) `num_actions` simultaneous actions."""

    actions_str = ", ".join(action_list)
    definitions_str = "\n".join(f"- {a}: {ACTION_DESCRIPTIONS[a]}" for a in action_list if a in ACTION_DESCRIPTIONS)

    objects_line = ""
    if object_names:
        objects_line = "Objects near the actor (drawn in BLUE boxes): " + ", ".join(object_names) + ".\n"

    prompt_text = (
        "You are an expert action recognition system. Analyze the provided video sequence.\n"
        "The primary actor you must analyze is enclosed in a RED bounding box. "
        "Other interactable objects are in BLUE bounding boxes.\n"
        f"{objects_line}"
        "The person can perform two actions at the same time, typically one with the right arm "
        "and one with the left arm. Identify the TWO actions being performed by the person in the RED box "
        "(one per arm), ordered from the most to the least prominent.\n"
        f"You MUST choose each action from this exact list: [{actions_str}].\n"
        "Action definitions:\n"
        f"{definitions_str}\n\n"
        f"If both arms are performing the same action, repeat it. "
        f"Respond with ONLY exactly {num_actions} action names from the list, separated by a comma "
        "(example: Reaching, TakingSomething). Do not add any other text."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": drawn_frame_paths,
                    "max_pixels": max_pixels,
                    "sample_fps": EFFECTIVE_FPS
                },
                {
                    "type": "text",
                    "text": prompt_text
                }
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True
    )

    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs = list(video_inputs)
        video_metadatas = list(video_metadatas)
    else:
        video_metadatas = None

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs
    ).to("cuda")

    # CamelCase names are split in many tokens: ~12 tokens per action is a safe upper bound
    max_new_tokens = 16 * num_actions

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False # Greedy decoding
        )

        # isolate output
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

    del inputs, generated_ids
    torch.cuda.empty_cache()

    # Validate output
    predicted_actions = parse_actions(output_text, action_list, max_actions=num_actions)
    return predicted_actions, output_text

# MAIN LOOP
video_folders = glob.glob(os.path.join(PATH_OBJ_DET_FINISHED, "*"))

action_labeling_start = time.time()
processed_videos = 0
processed_people = 0

for video_folder in video_folders:
    video_name = os.path.basename(video_folder)
    print(f"\n[+] Processing video: {video_name}")

    json_files = glob.glob(os.path.join(video_folder, "*.json"))
    if not json_files:
        print(f"  [Skip] No JSON found for {video_name}")
        continue

    with open(json_files[0], 'r') as f:
        tracking_data = json.load(f)

    detected_objects = tracking_data.get("detected_objects", {})
    people_ids = [k for k in detected_objects.keys() if k.startswith("person_")]

    if not people_ids:
        print(f"  [Skip] No actor found in JSON of {video_name}")
        continue

    # FILE CHECK
    people_to_process = []
    for person_id in people_ids:
        output_filename = f"{video_name}_{person_id}_actions.json"
        output_file_path = os.path.join(PATH_OUTPUT_DATASET, output_filename)

        if os.path.exists(output_file_path):
            print(f"  [Skip] Already processed: {person_id}")
        else:
            people_to_process.append((person_id, output_file_path))

    # skip to next video if all actors are processed
    if not people_to_process:
        print(f"  [Skip] Video {video_name} processed.")
        continue

    # PERSON-OBJECT ASSOCIATIONS
    association_data = load_associations(video_name)
    if association_data is None:
        print(f"  [Skip] No association file for {video_name} in {PATH_ASSOCIATIONS} "
              f"(run the person-object association step first)")
        continue
    associated_frames = build_associated_frames(association_data)
    # same acceptance thresholds used when the associations were produced
    association_meta = association_data.get("metadata", {})
    min_support_ratio = association_meta.get("minimum_support_ratio", 0.50)
    min_consecutive_frames = association_meta.get("minimum_consecutive_frames", 5)

    # FRAME LOADING
    frames_dir = os.path.join(PATH_VIDEOS_CROP_DECODE, video_name)
    all_frames = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
    total_frames = len(all_frames)

    if total_frames == 0:
        print(f"  [Skip] No frame found {video_name}")
        continue

    # process remaining actors
    processed_videos += 1
    for person_id, output_file_path in people_to_process:
        processed_people += 1
        print(f"  Action analysis for: {person_id}")
        person_bboxes = detected_objects[person_id].get("bbox", [])
        person_action_log = {}

        windows = build_windows(total_frames, WINDOW_SIZE, STRIDE, INCLUDE_TAIL_WINDOW)

        for start_idx, end_idx in windows:
            window_frames = all_frames[start_idx:end_idx]
            sample_indices = np.linspace(0, len(window_frames) - 1, min(FRAMES_TO_SAMPLE, len(window_frames)), dtype=int)

            sampled_frames_paths = [window_frames[i] for i in sample_indices]

            # Extract bbox
            sampled_person_bboxes = [
                person_bboxes[start_idx + i] if (start_idx + i) < len(person_bboxes) else None
                for i in sample_indices
            ]

            # skip if the actor isn't enough in the scene
            if sampled_person_bboxes.count(None) > (FRAMES_TO_SAMPLE / 2):
                continue

            sampled_objects_bboxes = []
            window_object_names = []

            # only the objects associated to THIS person in this window
            associated_object_ids = get_associated_objects(
                person_id, start_idx, end_idx, detected_objects, associated_frames,
                min_support_ratio, min_consecutive_frames
            )

            for obj_key in associated_object_ids:
                obj_data = detected_objects[obj_key]
                obj_boxes = obj_data.get("bbox", [])
                s_obj_boxes = [
                    obj_boxes[start_idx + i] if (start_idx + i) < len(obj_boxes) else None
                    for i in sample_indices
                ]

                if any(box is not None for box in s_obj_boxes):
                    sampled_objects_bboxes.append(s_obj_boxes)

                    # names of the associated objects (no duplicates)
                    obj_name = get_object_name(obj_key, obj_data)
                    if obj_name not in window_object_names:
                        window_object_names.append(obj_name)

            window_object_names = window_object_names[:MAX_OBJECT_NAMES]

            # Optional crop (same for all the frames of the window)
            crop_box = compute_actor_crop(sampled_person_bboxes) if CROP_MODE == "actor" else None

            # Temporary directory for bboxes
            temp_dir = tempfile.mkdtemp()
            try:
                drawn_paths = draw_bboxes_on_frames(
                    sampled_frames_paths,
                    sampled_person_bboxes,
                    sampled_objects_bboxes,
                    temp_dir,
                    crop_box=crop_box
                )

                try:
                    predicted_actions, raw_output = process_window(
                        drawn_frame_paths=drawn_paths,
                        action_list=CONSENTED_ACTIONS,
                        object_names=window_object_names
                    )
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    print(f"    [OOM] Try again with lower resolution {start_idx}-{end_idx}")
                    torch.cuda.empty_cache()
                    gc.collect()
                    predicted_actions, raw_output = process_window(
                        drawn_frame_paths=drawn_paths,
                        action_list=CONSENTED_ACTIONS,
                        object_names=window_object_names,
                        max_pixels=320*240
                    )

                timestamp_key = f"frames_{start_idx}_to_{end_idx}"
                person_action_log[timestamp_key] = {"actions": predicted_actions}
                print(f"    -> Window {start_idx}-{end_idx}: {predicted_actions}  objects={window_object_names}  (raw: '{raw_output}')")

            finally:
                shutil.rmtree(temp_dir, ignore_errors=True)

        output_filename = f"{video_name}_{person_id}_actions.json"
        output_file_path = os.path.join(PATH_OUTPUT_DATASET, output_filename)

        with open(output_file_path, 'w', encoding='utf-8') as out_f:
            json.dump(person_action_log, out_f, indent=4, ensure_ascii=False)

        print(f"  Saving complete: {output_file_path}")
        gc.collect()


action_labeling_time = time.time() - action_labeling_start
print(f'\n[+] Pipeline completed for all videos.')
print(f'\n[+] Total time of action recognition: {action_labeling_time} ')
avg_time = action_labeling_time / max(processed_videos, 1)
avg_human_time = action_labeling_time / max(processed_people, 1)
print(f'\n[+] Avg time per video: {avg_time:.1f}s')
print(f'\n[+] Avg time per human: {avg_human_time:.1f}s')

In [ ]:
!cd /content/action_recognition_finished && zip -r -q /content/action_recognition_finished.zip .

In [ ]:
!cp "/content/action_recognition_finished.zip" "/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/"